# Курсовая работа: Выдача рейтинга фильма

## Инициализация и проверка

In [16]:
import torch
import sys
import json
import warnings
warnings.filterwarnings('ignore')
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

PyTorch version: 2.9.1+cu130
CUDA available: True
GPU: NVIDIA GeForce GTX 1650
CUDA version: 13.0


In [17]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

CUDA: True
NVIDIA GeForce GTX 1650


## Конфиг (параметры обучения)

In [18]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import os

# Конфигурация путей
ROOT = Path(".").resolve()
OUTPUT_DIR = ROOT / "Out"
DATA_DIR = ROOT / "datasets"
METADATA_CSV = DATA_DIR / "scripts_ratingss.csv"
RAW_DATA_DIR = DATA_DIR / "scenaryy"
TEST_NOINFO_DIR = DATA_DIR / "noinfo"
TEST_GOOD_DIR = DATA_DIR / "good"

# Создание директорий
for dir_path in [OUTPUT_DIR, TEST_NOINFO_DIR, TEST_GOOD_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print(f"\nРабочая директория: {ROOT}")
print(f"Директория с данными: {DATA_DIR}")


Рабочая директория: C:\Users\Дмитрий\Desktop\work\Ratings
Директория с данными: C:\Users\Дмитрий\Desktop\work\Ratings\datasets


## Загрузка и проверка данных

In [19]:
class DataLoader:
    @staticmethod
    def load_metadata(csv_path: Path) -> pd.DataFrame:
        """Загрузка метаданных из CSV"""
        try:
            df = pd.read_csv(csv_path, encoding='utf-8')
            print(f"Загружено {len(df)} записей из {csv_path.name}")
            return df
        except Exception as e:
            raise Exception(f"Ошибка загрузки CSV: {e}")
    
    @staticmethod
    def load_script_txt(file_path: Path) -> str:
        """Загрузка текста сценария с обработкой кодировок"""
        encodings = ['utf-8', 'cp1251', 'iso-8859-1', 'windows-1251']
        for encoding in encodings:
            try:
                with open(file_path, 'r', encoding=encoding) as f:
                    return f.read()
            except UnicodeDecodeError:
                continue
        raise Exception(f"Не удалось декодировать файл: {file_path.name}")

class TextPreprocessor:
    @staticmethod
    def clean_script_text(raw_text: str) -> str:
        """Очистка текста сценария"""
        text = raw_text
        
        # Удаление технической информации
        text = re.sub(r'\\r\\n', '\n', text)
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'[ \t]{2,}', ' ', text)
        
        # Удаление скобочных комментариев
        text = re.sub(r'\([^)]*\)', '', text)
        
        # Удаление строк только из заглавных букв (ремарки)
        text = re.sub(r'^[A-ZА-ЯЁ\s\d\-\.]+$', '', text, flags=re.MULTILINE)
        
        # Очистка от лишних символов
        text = re.sub(r'[^\w\s\.,!?;:\-\'"\n]', '', text)
        
        # Нормализация пробелов вокруг знаков препинания
        text = re.sub(r'\s+([.,!?;:])', r'\1', text)
        text = re.sub(r'([.,!?;:])\s+', r'\1 ', text)
        
        return text.strip()
    
    @staticmethod
    def split_text_for_bert(text: str, max_chunk_size: int = 4000) -> list:
        """Разбивка длинного текста на части для BERT"""
        words = text.split()
        chunks = []
        current_chunk = []
        current_length = 0
        
        for word in words:
            if current_length + len(word.split()) > max_chunk_size and current_chunk:
                chunks.append(' '.join(current_chunk))
                current_chunk = []
                current_length = 0
            current_chunk.append(word)
            current_length += len(word.split())
        
        if current_chunk:
            chunks.append(' '.join(current_chunk))
        
        return chunks

# Загрузка метаданных
metadata_df = DataLoader.load_metadata(METADATA_CSV)
print("\nПервые 5 записей:")
print(metadata_df.head().to_string())

Загружено 51 записей из scripts_ratingss.csv

Первые 5 записей:
                    filename                  title  year  kp_rating  imdb_rating notes age_rating_imdb age_rating_kp           english_title
0  13_причин_почему_Кино.txt  13_причин_почему_Кино  2017        7.3          7.4    ok           TV-MA           16+    TH1RTEEN R3ASONS WHY
1    Игра_Престолов_Кино.txt    Игра_Престолов_Кино  2011        9.0          9.2    ok           TV-MA           18+  A Song of Ice and Fire
2     8_миллиметров_Кино.txt     8_миллиметров_Кино  1999        7.1          6.6    ok               R           18+            8 Millimeter
3       Джентльмены_Кино.txt       Джентльмены_Кино  2019        8.7          7.8    ok               R           18+           The Gentlemen
4            Джокер_Кино.txt            Джокер_Кино  2019        8.0          8.3    ok               R           18+                   Joker


## Подготовка к обучению

In [20]:
def prepare_training_dataset(metadata_df, raw_data_dir):
    """Подготовка датасета для обучения"""
    processed_data = []
    
    for idx, row in metadata_df.iterrows():
        try:
            filename = str(row['filename'])
            if not filename.endswith('.txt'):
                filename += '.txt'
            
            txt_path = raw_data_dir / filename
            
            if not txt_path.exists():
                print(f"Файл не найден: {filename}")
                continue
            
            # Загрузка и очистка текста
            raw_text = DataLoader.load_script_txt(txt_path)
            cleaned_text = TextPreprocessor.clean_script_text(raw_text)
            
            if len(cleaned_text) < 1000:
                print(f"Текст слишком короткий: {filename} ({len(cleaned_text)} символов)")
                continue
            
            # Подготовка данных
            record = {
                'filename': filename,
                'title': row['title'],
                'year': int(row['year']) if pd.notna(row['year']) else 0,
                'kp_rating': float(row['kp_rating']) if pd.notna(row['kp_rating']) else 0.0,
                'imdb_rating': float(row['imdb_rating']) if pd.notna(row['imdb_rating']) else 0.0,
                'age_rating': str(row['age_rating_kp']) if pd.notna(row['age_rating_kp']) else str(row['age_rating_imdb']),
                'text': cleaned_text
            }
            
            # Обработка возрастного рейтинга
            age_rating = record['age_rating']
            if age_rating in ['R', 'TV-MA', '18+']:
                record['age_label'] = 4  # 18+
            elif age_rating in ['16+', 'TV-14']:
                record['age_label'] = 3  # 16+
            elif age_rating in ['12+', 'PG-13']:
                record['age_label'] = 2  # 12+
            elif age_rating in ['6+', 'PG']:
                record['age_label'] = 1  # 6+
            elif age_rating in ['0+', 'G']:
                record['age_label'] = 0  # 0+
            else:
                record['age_label'] = 2  # По умолчанию 12+
            
            processed_data.append(record)
            
        except Exception as e:
            print(f"Ошибка обработки файла {filename}: {e}")
            continue
    
    return pd.DataFrame(processed_data)

# Создание датасета
print("\nПодготовка датасета...")
processed_df = prepare_training_dataset(metadata_df, RAW_DATA_DIR)
print(f"Создан датасет: {len(processed_df)} записей")
print(f"Средняя длина текста: {processed_df['text'].apply(len).mean():.0f} символов")


Подготовка датасета...
Создан датасет: 51 записей
Средняя длина текста: 76027 символов


In [21]:
from sklearn.model_selection import train_test_split
import numpy as np

# Проверяем распределение по классам age_label
print("\nРаспределение по возрастным рейтингам:")
age_distribution = processed_df['age_label'].value_counts().sort_index()
print(age_distribution)

# Проверяем, есть ли классы с малым количеством примеров
min_samples = age_distribution.min()
print(f"\nМинимальное количество примеров в классе: {min_samples}")

# Если есть классы с 1-2 примерами, используем другую стратегию
if min_samples < 2:
    print("Некоторые классы имеют менее 2 примеров, используется простая случайная выборка без стратификации")
    
    # Разделение без стратификации
    train_df, val_df = train_test_split(
        processed_df,
        test_size=0.2,
        random_state=42
    )
else:
    print("Используется стратификация по возрастным рейтингам")
    
    # Разделение с стратификацией
    try:
        train_df, val_df = train_test_split(
            processed_df,
            test_size=0.2,
            random_state=42,
            stratify=processed_df['age_label']
        )
    except ValueError as e:
        print(f"Ошибка при стратификации: {e}")
        print("Используется альтернативная стратегия - стратификация по бинарному рейтингу")
        
        # Альтернатива: стратификация по бинарному рейтингу (выше/ниже медианы)
        median_rating = processed_df['kp_rating'].median()
        processed_df['rating_binary'] = (processed_df['kp_rating'] > median_rating).astype(int)
        
        train_df, val_df = train_test_split(
            processed_df,
            test_size=0.2,
            random_state=42,
            stratify=processed_df['rating_binary']
        )

print(f"\nРазделение данных:")
print(f"Тренировочная выборка: {len(train_df)} записей")
print(f"Валидационная выборка: {len(val_df)} записей")

# Проверяем распределение классов в обеих выборках
print("\nРаспределение в тренировочной выборке:")
print(train_df['age_label'].value_counts().sort_index())

print("\nРаспределение в валидационной выборке:")
print(val_df['age_label'].value_counts().sort_index())


Распределение по возрастным рейтингам:
age_label
0     1
2     4
3     8
4    38
Name: count, dtype: int64

Минимальное количество примеров в классе: 1
Некоторые классы имеют менее 2 примеров, используется простая случайная выборка без стратификации

Разделение данных:
Тренировочная выборка: 40 записей
Валидационная выборка: 11 записей

Распределение в тренировочной выборке:
age_label
0     1
2     3
3     7
4    29
Name: count, dtype: int64

Распределение в валидационной выборке:
age_label
2    1
3    1
4    9
Name: count, dtype: int64


# BERT

In [22]:
print("ИНИЦИАЛИЗАЦИЯ МОДЕЛИ BERT")

# Импорт библиотек
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
# AdamW теперь импортируется из torch.optim
from torch.optim import AdamW
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, f1_score
import torch.nn as nn
import torch

ИНИЦИАЛИЗАЦИЯ МОДЕЛИ BERT


## Выбор модели

In [23]:
# Используем модель с большим количеством параметров
MODEL_NAME = "sberbank-ai/ruRoberta-large"  # 30M параметров, хорошо работает с русским
# Альтернативы с большим размером:
# "cointegrated/rubert-tiny2" - 355M параметров
# "DeepPavlov/rubert-base-cased" - 178M параметров

print(f"\nИспользуемая модель: {MODEL_NAME}")
print("Загрузка токенизатора...")

# Загрузка токенизатора
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)



Используемая модель: sberbank-ai/ruRoberta-large
Загрузка токенизатора...


## Класс датасета

In [24]:
class MovieDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        self.dataframe = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        text = str(self.dataframe.iloc[idx]['text'])
        rating = float(self.dataframe.iloc[idx]['kp_rating'])
        age_label = int(self.dataframe.iloc[idx]['age_label'])
        
        # Токенизация текста
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'rating_labels': torch.tensor(rating, dtype=torch.float32),
            'age_labels': torch.tensor(age_label, dtype=torch.long)
        }


## Многозадачность

In [25]:
# ==================================================
# МНОГОЗАДАЧНАЯ МОДЕЛЬ BERT С LoRA (ПРАВИЛЬНАЯ АРХИТЕКТУРА)
# ==================================================
from transformers import AutoModel
from peft import LoraConfig, get_peft_model, TaskType

class MultitaskBERTModelWithLoRA(nn.Module):
    def __init__(self, base_model_name, num_age_classes=5, use_lora=True):
        super().__init__()
        
        # Базовая модель BERT/RoBERTa без головы
        self.base_model = AutoModel.from_pretrained(
            base_model_name,
            output_attentions=False,
            output_hidden_states=False
        )
        
        # Применение LoRA к базовой модели
        if use_lora:
            lora_config = LoraConfig(
                r=16,
                lora_alpha=32,
                target_modules=["query", "key", "value", "dense"],
                lora_dropout=0.1,
                bias="none",
                task_type=TaskType.FEATURE_EXTRACTION,
                inference_mode=False
            )
            self.base_model = get_peft_model(self.base_model, lora_config)
        
        # Получаем размер скрытого слоя
        hidden_size = self.base_model.config.hidden_size
        
        # Голова для регрессии рейтинга
        self.rating_head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)
        )
        
        # Голова для классификации возрастного рейтинга
        self.age_head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_age_classes)
        )
        
        self._init_weights()
    
    def _init_weights(self):
        """Инициализация весов голов"""
        for head in [self.rating_head, self.age_head]:
            for module in head.modules():
                if isinstance(module, nn.Linear):
                    nn.init.xavier_uniform_(module.weight)
                    if module.bias is not None:
                        nn.init.zeros_(module.bias)
    
    def forward(self, input_ids=None, attention_mask=None, rating_labels=None, age_labels=None, **kwargs):
        # Прямой проход через базовую модель
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        # Используем средний пулинг по токенам
        pooled_output = outputs.last_hidden_state.mean(dim=1)
        
        # Прямой проход через головы
        rating_logits = self.rating_head(pooled_output).squeeze(-1)
        age_logits = self.age_head(pooled_output)
        
        # Вычисление потерь
        loss = None
        if rating_labels is not None and age_labels is not None:
            rating_loss = nn.MSELoss()(rating_logits, rating_labels)
            age_loss = nn.CrossEntropyLoss()(age_logits, age_labels.long())
            loss = rating_loss + 0.5 * age_loss
        
        # Возвращаем словарь с необходимыми полями
        return {
            'loss': loss,
            'logits': rating_logits,
            'rating_logits': rating_logits,
            'age_logits': age_logits,
            'hidden_states': outputs.hidden_states,
            'attentions': outputs.attentions
        }

In [26]:
# ==================================================
# КАСТОМНЫЙ TRAINER ДЛЯ МНОГОЗАДАЧНОГО ОБУЧЕНИЯ
# ==================================================
from transformers import Trainer
import torch

class MultitaskTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
    
    def compute_loss(self, model, inputs, return_outputs=False):
        """
        Переопределяем вычисление потерь для многозадачного обучения
        """
        # Извлекаем данные
        input_ids = inputs.get('input_ids')
        attention_mask = inputs.get('attention_mask')
        rating_labels = inputs.get('rating_labels')
        age_labels = inputs.get('age_labels')
        
        # Прямой проход через модель
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            rating_labels=rating_labels,
            age_labels=age_labels
        )
        
        loss = outputs['loss']
        
        if return_outputs:
            return loss, outputs
        
        return loss
    
    def prediction_step(self, model, inputs, prediction_loss_only=False, ignore_keys=None):
        """
        Переопределяем шаг предсказания для многозадачной модели
        """
        # Удаляем labels из inputs, чтобы они не мешали
        inputs_no_labels = {k: v for k, v in inputs.items() if k not in ['rating_labels', 'age_labels', 'labels']}
        
        with torch.no_grad():
            outputs = model(**inputs_no_labels)
            loss = outputs['loss']
            
            # Подготавливаем predictions для compute_metrics
            predictions = (outputs['rating_logits'], outputs['age_logits'])
            
            # Подготавливаем labels для compute_metrics
            labels = (inputs.get('rating_labels'), inputs.get('age_labels'))
        
        if prediction_loss_only:
            return (loss.detach(), None, None)
        
        return (loss.detach(), predictions, labels)

# Метрики и датасеты

In [27]:
def compute_multitask_metrics(eval_pred):
    """
    Вычисление метрик для многозадачного обучения
    """
    predictions, labels = eval_pred
    
    # Извлекаем предсказания
    rating_preds = predictions[0]  # Первый элемент - рейтинги
    age_preds = predictions[1]     # Второй элемент - возрастные рейтинги
    
    # Извлекаем истинные значения
    rating_labels = labels[0]
    age_labels = labels[1]
    
    # Метрики для рейтинга (регрессия)
    rating_mse = mean_squared_error(rating_labels, rating_preds)
    rating_mae = mean_absolute_error(rating_labels, rating_preds)
    rating_rmse = np.sqrt(rating_mse)
    
    # Метрики для возрастного рейтинга (классификация)
    age_preds_class = np.argmax(age_preds, axis=1)
    age_accuracy = accuracy_score(age_labels, age_preds_class)
    age_precision = precision_score(age_labels, age_preds_class, average='weighted', zero_division=0)
    age_recall = recall_score(age_labels, age_preds_class, average='weighted', zero_division=0)
    age_f1 = f1_score(age_labels, age_preds_class, average='weighted', zero_division=0)
    
    return {
        'rating_mse': float(rating_mse),
        'rating_mae': float(rating_mae),
        'rating_rmse': float(rating_rmse),
        'age_accuracy': float(age_accuracy),
        'age_precision': float(age_precision),
        'age_recall': float(age_recall),
        'age_f1': float(age_f1)
    }

In [28]:
print("\nПодготовка датасетов для обучения...")

train_dataset = MovieDataset(train_df, tokenizer, max_length=512)
val_dataset = MovieDataset(val_df, tokenizer, max_length=512)

print(f"Размер тренировочного датасета: {len(train_dataset)}")
print(f"Размер валидационного датасета: {len(val_dataset)}")


Подготовка датасетов для обучения...
Размер тренировочного датасета: 40
Размер валидационного датасета: 11


## Моделька

In [29]:
# ==================================================
# ИНИЦИАЛИЗАЦИЯ МОДЕЛИ И TRAINER
# ==================================================
print("\nИнициализация модели с LoRA...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultitaskBERTModelWithLoRA(MODEL_NAME, use_lora=True).to(device)

# Вывод информации о модели
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Всего параметров: {total_params:,}")
print(f"Обучаемых параметров (с LoRA): {trainable_params:,}")
print(f"Процент обучаемых параметров: {trainable_params/total_params*100:.2f}%")

# Параметры обучения
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR / "bert_lora_model",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir=OUTPUT_DIR / "logs",
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="rating_mae",
    greater_is_better=False,
    report_to="none",
    fp16=torch.cuda.is_available(),
    gradient_accumulation_steps=4,
)

# Создаем кастомный Trainer
trainer = MultitaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_multitask_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)


Инициализация модели с LoRA...


Some weights of RobertaModel were not initialized from the model checkpoint at sberbank-ai/ruRoberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Всего параметров: 362,996,742
Обучаемых параметров (с LoRA): 7,636,998
Процент обучаемых параметров: 2.10%


### Обучалка

In [30]:
train_results = trainer.train()

# Сохранение модели
trainer.save_model()
tokenizer.save_pretrained(OUTPUT_DIR / "bert_lora_model")
print(f"\nМодель сохранена в: {OUTPUT_DIR / 'bert_lora_model'}")

# Сохранение полной модели (не только адаптеров LoRA)
torch.save(model.state_dict(), OUTPUT_DIR / "bert_lora_model" / "pytorch_model.bin")

TypeError: MultitaskTrainer.compute_loss() got an unexpected keyword argument 'num_items_in_batch'

In [ ]:
print("ОЦЕНКА НА ВАЛИДАЦИОННОЙ ВЫБОРКЕ")

# Предсказания на валидационной выборке
val_predictions = trainer.predict(val_dataset)

# Извлечение предсказаний
rating_preds = val_predictions.predictions[0].flatten()
age_preds = np.argmax(val_predictions.predictions[1], axis=1)

# Фактические значения
rating_labels = val_df['kp_rating'].values
age_labels = val_df['age_label'].values

# Создание DataFrame с результатами
results_df = pd.DataFrame({
    'title': val_df['title'],
    'actual_rating': rating_labels,
    'predicted_rating': rating_preds,
    'rating_error': np.abs(rating_labels - rating_preds),
    'actual_age': age_labels,
    'predicted_age': age_preds,
    'age_correct': (age_labels == age_preds).astype(int)
})

# Вывод таблицы с результатами
print("\nРезультаты валидации:")
print(f"{'Фильм':<30} {'Реальный':<10} {'Предсказ.':<10} {'Ошибка':<8} {'Возраст':<8} {'Совпад.'}")
print("-" * 80)

for idx, row in results_df.iterrows():
    age_match = "✓" if row['age_correct'] else "✗"
    print(f"{row['title'][:28]:<30} {row['actual_rating']:<10.2f} {row['predicted_rating']:<10.2f} "
          f"{row['rating_error']:<8.2f} {row['predicted_age']:<8} {age_match:<8}")

# Статистика
print("СТАТИСТИКА ВАЛИДАЦИИ")

rating_mae = mean_absolute_error(rating_labels, rating_preds)
rating_rmse = np.sqrt(mean_squared_error(rating_labels, rating_preds))
age_accuracy = accuracy_score(age_labels, age_preds)

metrics_table = pd.DataFrame({
    'Метрика': ['MAE рейтинга', 'RMSE рейтинга', 'Точность возрастного рейтинга'],
    'Значение': [f"{rating_mae:.3f}", f"{rating_rmse:.3f}", f"{age_accuracy:.2%}"],
    'Описание': ['Средняя абсолютная ошибка', 'Корень из среднеквадратичной ошибки', 'Доля верных предсказаний']
})

print(metrics_table.to_string(index=False))

# Сохранение результатов
results_path = OUTPUT_DIR / "validation_results.csv"
results_df.to_csv(results_path, index=False, encoding='utf-8')
print(f"\nРезультаты сохранены в: {results_path}")


# Тесты

In [ ]:
# ==================================================
# ТЕСТИРОВАНИЕ НА ПАПКЕ NOINFO
# ==================================================
print("\n" + "="*60)
print("ТЕСТИРОВАНИЕ НА ПАПКЕ NOINFO")
print("="*60)

def predict_on_unlabeled_files(test_dir, model, tokenizer, device):
    """Предсказание на файлах без разметки"""
    results = []
    test_files = list(test_dir.glob("*.txt"))
    
    print(f"Найдено файлов: {len(test_files)}")
    
    for test_file in test_files:
        try:
            # Загрузка и очистка текста
            raw_text = DataLoader.load_script_txt(test_file)
            cleaned_text = TextPreprocessor.clean_script_text(raw_text)
            
            if len(cleaned_text) < 100:
                print(f"Файл слишком короткий: {test_file.name}")
                continue
            
            # Токенизация
            encoding = tokenizer(
                cleaned_text,
                truncation=True,
                padding='max_length',
                max_length=512,
                return_tensors='pt'
            )
            
            # Перенос на устройство
            input_ids = encoding['input_ids'].to(device)
            attention_mask = encoding['attention_mask'].to(device)
            
            # Предсказание
            model.eval()
            with torch.no_grad():
                outputs = model(input_ids, attention_mask)
            
            # Извлечение результатов
            rating_pred = outputs['rating_predictions'].item()
            age_pred = outputs['age_predictions'].item()
            
            # Преобразование возрастного рейтинга
            age_mapping = {0: "0+", 1: "6+", 2: "12+", 3: "16+", 4: "18+"}
            age_rating = age_mapping.get(age_pred, "12+")
            
            results.append({
                'filename': test_file.name,
                'predicted_rating': round(rating_pred, 2),
                'predicted_age_rating': age_rating,
                'text_length': len(cleaned_text)
            })
            
            print(f"Обработан: {test_file.name}")
            
        except Exception as e:
            print(f"Ошибка при обработке {test_file.name}: {e}")
    
    return pd.DataFrame(results)

# Предсказание на noinfo
if TEST_NOINFO_DIR.exists():
    noinfo_results = predict_on_unlabeled_files(TEST_NOINFO_DIR, model, tokenizer, device)
    
    if len(noinfo_results) > 0:
        print("\nРезультаты тестирования (noinfo):")
        print("-" * 70)
        print(f"{'Файл':<30} {'Рейтинг':<10} {'Возраст.':<10} {'Длина':<10}")
        print("-" * 70)
        
        for idx, row in noinfo_results.iterrows():
            print(f"{row['filename'][:28]:<30} {row['predicted_rating']:<10.2f} "
                  f"{row['predicted_age_rating']:<10} {row['text_length']:<10}")
        
        # Сохранение результатов
        noinfo_path = OUTPUT_DIR / "noinfo_predictions.csv"
        noinfo_results.to_csv(noinfo_path, index=False, encoding='utf-8')
        print(f"\nРезультаты сохранены в: {noinfo_path}")
else:
    print("Папка noinfo не найдена")

# ==================================================
# ТЕСТИРОВАНИЕ НА ПАПКЕ GOOD
# ==================================================
print("\n" + "="*60)
print("ТЕСТИРОВАНИЕ НА ПАПКЕ GOOD")
print("="*60)

if TEST_GOOD_DIR.exists():
    good_results = predict_on_unlabeled_files(TEST_GOOD_DIR, model, tokenizer, device)
    
    if len(good_results) > 0:
        print("\nРезультаты тестирования (good):")
        print("-" * 70)
        print(f"{'Файл':<30} {'Рейтинг':<10} {'Возраст.':<10} {'Длина':<10}")
        print("-" * 70)
        
        for idx, row in good_results.iterrows():
            print(f"{row['filename'][:28]:<30} {row['predicted_rating']:<10.2f} "
                  f"{row['predicted_age_rating']:<10} {row['text_length']:<10}")
        
        # Сохранение результатов
        good_path = OUTPUT_DIR / "good_predictions.csv"
        good_results.to_csv(good_path, index=False, encoding='utf-8')
        print(f"\nРезультаты сохранены в: {good_path}")
else:
    print("Папка good не найдена")

# ==================================================
# СОХРАНЕНИЕ КОНФИГУРАЦИИ И МЕТРИК
# ==================================================
config = {
    "model_name": MODEL_NAME,
    "total_parameters": f"{total_params:,}",
    "trainable_parameters": f"{trainable_params:,}",
    "training_samples": len(train_df),
    "validation_samples": len(val_df),
    "test_samples_noinfo": len(noinfo_results) if 'noinfo_results' in locals() else 0,
    "test_samples_good": len(good_results) if 'good_results' in locals() else 0,
    "training_date": pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    "validation_metrics": {
        "rating_mae": float(rating_mae),
        "rating_rmse": float(rating_rmse),
        "age_accuracy": float(age_accuracy)
    },
    "model_architecture": "Multitask BERT with regression and classification heads"
}

with open(OUTPUT_DIR / "training_config.json", "w", encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("\n" + "="*60)
print("КОНФИГУРАЦИЯ СОХРАНЕНА")
print("="*60)
print(json.dumps(config, indent=2, ensure_ascii=False))


# TRY 2

In [1]:
# ==================================================
# ИНИЦИАЛИЗАЦИЯ И ПРОВЕРКА CUDA
# ==================================================
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

# ==================================================
# НАСТРОЙКА ПУТЕЙ И КОНФИГУРАЦИЯ
# ==================================================
from pathlib import Path
import pandas as pd
import re
import os
import warnings
from typing import Dict, List, Optional

# Конфигурация путей
ROOT = Path(".").resolve() 
OUTPUT_DIR = ROOT / "Out"
DATA_DIR = ROOT / "datasets"
METADATA_CSV = DATA_DIR / "scripts_ratingss.csv"
PROCESSED_DIR = DATA_DIR / "Pipe"
RAW_DATA_DIR = DATA_DIR / "scenaryy"
TEST_TXT_DIR = DATA_DIR / "noinfo"

# Параметры
MIN_TEXT_LENGTH = 1000
MAX_TEXT_LENGTH = 100000
ENCODING = 'utf-8'

# Создание директорий
for dir_path in [RAW_DATA_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print("Конфигурация загружена!")
print(f"ROOT: {ROOT}")

# ==================================================
# ЗАГРУЗКА И ПРЕДОБРАБОТКА ДАННЫХ
# ==================================================
class DataLoader:
    @staticmethod
    def load_metadata(csv_path: Path) -> pd.DataFrame:
        try:
            df = pd.read_csv(csv_path, encoding=ENCODING)
            print(f"Загружено {len(df)} записей из {csv_path.name}")
            return df
        except Exception as e:
            raise Exception(f"Ошибка загрузки CSV: {e}")
    
    @staticmethod
    def load_script_txt(file_path: Path) -> str:
        try:
            with open(file_path, 'r', encoding=ENCODING) as f:
                text = f.read()
            return text
        except UnicodeDecodeError:
            for encoding in ['cp1251', 'iso-8859-1', 'mac_cyrillic']:
                try:
                    with open(file_path, 'r', encoding=encoding) as f:
                        return f.read()
                except:
                    continue
            raise Exception(f"Не удалось декодировать файл: {file_path.name}")

class TextPreprocessor:
    @staticmethod
    def clean_script_text(raw_text: str) -> str:
        text = raw_text
        text = re.sub(r'\r\n', '\n', text)
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'[ \t]{2,}', ' ', text)
        text = re.sub(r'\([^)]*\)', '', text)
        text = re.sub(r'^[A-ZА-Я\s\d\-\.]+$', '', text, flags=re.MULTILINE)
        text = re.sub(r'[^\w\s\.,!?;:\-\'\"\(\)\n]', '', text)
        text = re.sub(r'\s+([.,!?;:])', r'\1', text)
        text = re.sub(r'([.,!?;:])\s+', r'\1 ', text)
        return text.strip()

# Загрузка метаданных
metadata_df = DataLoader.load_metadata(METADATA_CSV)
print("\nПервые 3 записи:")
print(metadata_df.head(3))

# ==================================================
# ПОДГОТОВКА ДАТАСЕТА ДЛЯ ОБУЧЕНИЯ (ПОЛНЫЙ СЦЕНАРИЙ!)
# ==================================================
def prepare_training_data(metadata_df, raw_data_dir, processed_dir):
    processed_data = []
    
    for idx, row in metadata_df.iterrows():
        try:
            filename = str(row['filename'])
            if not filename.endswith('.txt'):
                filename += '.txt'
            
            txt_path = raw_data_dir / filename
            
            if not txt_path.exists():
                print(f"Файл не найден: {filename}")
                continue
            
            # Загрузка и очистка текста (ВЕСЬ текст!)
            raw_text = DataLoader.load_script_txt(txt_path)
            cleaned_text = TextPreprocessor.clean_script_text(raw_text)
            
            if len(cleaned_text) < MIN_TEXT_LENGTH:
                print(f"Текст слишком короткий: {filename} ({len(cleaned_text)} символов)")
                continue
            
            # Используем ВЕСЬ текст для обучения
            # Но обрежем до максимальной длины, если нужно
            if len(cleaned_text) > MAX_TEXT_LENGTH:
                cleaned_text = cleaned_text[:MAX_TEXT_LENGTH]
                print(f"Обрезан длинный текст: {filename} ({len(cleaned_text)} символов)")
            
            # Создание записи для обучения
            record = {
                'filename': filename,
                'title': row['title'],
                'year': row['year'],
                'kp_rating': row['kp_rating'],
                'imdb_rating': row['imdb_rating'],
                'age_rating': row['age_rating_kp'] if pd.notna(row['age_rating_kp']) else row['age_rating_imdb'],
                'text': cleaned_text  # ВЕСЬ текст!
            }
            processed_data.append(record)
            
        except Exception as e:
            print(f"Ошибка обработки файла {filename}: {e}")
            continue
    
    return pd.DataFrame(processed_data)

# Создание датасета
processed_df = prepare_training_data(metadata_df, RAW_DATA_DIR, PROCESSED_DIR)
print(f"\nСоздан датасет: {len(processed_df)} записей")
print(f"Средняя длина текста: {processed_df['text'].apply(len).mean():.0f} символов")

# ==================================================
# РАЗДЕЛЕНИЕ НА ТРЕНИРОВОЧНУЮ И ТЕСТОВУЮ ВЫБОРКИ
# ==================================================
from sklearn.model_selection import train_test_split

# Создание бинарных меток для рейтинга (выше/ниже медианы)
median_rating = processed_df['kp_rating'].median()
processed_df['rating_label'] = (processed_df['kp_rating'] > median_rating).astype(int)

# Преобразование возрастного рейтинга в числовые метки
age_mapping = {
    '18+': 0,
    '16+': 1,
    '12+': 2,
    '6+': 3,
    '0+': 4,
    'Not found': 5,
    'NOT_RATED': 5,
    'NR': 5,
    'R': 0  # R примерно соответствует 18+
}
processed_df['age_label'] = processed_df['age_rating'].map(age_mapping).fillna(5).astype(int)

# Разделение данных
train_df, val_df = train_test_split(
    processed_df,
    test_size=0.15,  # Уменьшил для большего тренировочного набора
    random_state=42,
    stratify=processed_df['rating_label']
)

print(f"\nРазделение данных:")
print(f"Тренировочная выборка: {len(train_df)} записей")
print(f"Валидационная выборка: {len(val_df)} записей")

# ==================================================
# УЛУЧШЕННОЕ СОЗДАНИЕ ПРОМПТОВ ДЛЯ ОБУЧЕНИЯ
# ==================================================
def create_prompts(df, task_type="both"):
    """Создание улучшенных промптов для обучения"""
    prompts = []
    
    for idx, row in df.iterrows():
        # Берем первые 15000 символов для промпта (можно увеличить)
        text_for_prompt = row['text'][:15000]
        
        if task_type == "both":
            # УЛУЧШЕННЫЙ промпт для предсказания обоих показателей
            prompt = f"""ТЕКСТ СЦЕНАРИЯ ФИЛЬМА:

{text_for_prompt}

ПРОАНАЛИЗИРУЙ ЭТОТ СЦЕНАРИЙ И ДАЙ ОЦЕНКУ:

РЕЙТИНГ (от 1.0 до 10.0, где 10.0 - отлично):
ВОЗРАСТНОЙ РЕЙТИНГ (выбери один: 0+, 6+, 12+, 16+, 18+):

ОТВЕТ:
РЕЙТИНГ = {row['kp_rating']:.1f}
ВОЗРАСТНОЙ РЕЙТИНГ = {row['age_rating']}"""
        
        elif task_type == "rating":
            # УЛУЧШЕННЫЙ промпт для предсказания рейтинга
            prompt = f"""ТЕКСТ СЦЕНАРИЯ ФИЛЬМА:

{text_for_prompt}

НА ОСНОВЕ ЭТОГО ТЕКСТА СЦЕНАРИЯ, ОЦЕНИ РЕЙТИНГ ФИЛЬМА ОТ 1.0 ДО 10.0 (ГДЕ 10.0 - ОТЛИЧНО):

ОТВЕТ: {row['kp_rating']:.1f}"""
            
        elif task_type == "age_rating":
            # УЛУЧШЕННЫЙ промпт для предсказания возрастного рейтинга
            prompt = f"""ТЕКСТ СЦЕНАРИЯ ФИЛЬМА:

{text_for_prompt}

ОПРЕДЕЛИ ВОЗРАСТНОЙ РЕЙТИНГ ДЛЯ ЭТОГО ФИЛЬМА (ВЫБЕРИ ОДИН ИЗ: 0+, 6+, 12+, 16+, 18+):

ОТВЕТ: {row['age_rating']}"""
        
        prompts.append(prompt)
    
    return prompts

# Создание УЛУЧШЕННЫХ промптов
train_prompts = create_prompts(train_df, "both")
val_prompts = create_prompts(val_df, "both")

# Сохранение промптов
prompts_dir = PROCESSED_DIR / "prompts"
prompts_dir.mkdir(exist_ok=True)

with open(prompts_dir / "train_prompts.txt", "w", encoding=ENCODING) as f:
    f.write("\n\n".join(train_prompts))

with open(prompts_dir / "val_prompts.txt", "w", encoding=ENCODING) as f:
    f.write("\n\n".join(val_prompts))

print(f"\nПромпты сохранены в: {prompts_dir}")

# ==================================================
# УСТАНОВКА БИБЛИОТЕК И ИНИЦИАЛИЗАЦИЯ МОДЕЛИ
# ==================================================
print("\n" + "="*50)
print("УСТАНОВКА И ЗАГРУЗКА МОДЕЛИ")
print("="*50)

# Импорт библиотек
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM,
    TrainingArguments, 
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
import torch

# ==================================================
# КОНФИГУРАЦИЯ МОДЕЛИ
# ==================================================
MODEL_NAME = "Qwen/Qwen2.5-1.5B"

# Конфигурация квантизации для экономии памяти
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Загрузка токенизатора
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    trust_remote_code=True
)

# Настройка токенизатора
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = 'right'

# ==================================================
# ПОДГОТОВКА ДАННЫХ ДЛЯ ОБУЧЕНИЯ
# ==================================================
def prepare_dataset_for_finetuning(prompts_list, max_length=2048):
    """Подготовка датасета для тонкой настройки"""
    dataset = Dataset.from_dict({"text": prompts_list})
    
    def tokenize_function(examples):
        # Токенизация с вниманием к длине
        result = tokenizer(
            examples["text"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt"
        )
        
        # Создаем labels (такие же как input_ids)
        result["labels"] = result["input_ids"].clone()
        
        return result
    
    tokenized_dataset = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=["text"]
    )
    
    return tokenized_dataset

# Подготовка датасетов с большей длиной
print("\nПодготовка датасетов...")
train_dataset = prepare_dataset_for_finetuning(train_prompts, max_length=2048)
val_dataset = prepare_dataset_for_finetuning(val_prompts, max_length=2048)

print(f"Размеры датасетов:")
print(f"Обучающий: {len(train_dataset)} примеров")
print(f"Валидационный: {len(val_dataset)} примеров")

# ==================================================
# НАСТРОЙКА И ОБУЧЕНИЕ МОДЕЛИ С LoRA
# ==================================================
print("\n" + "="*50)
print("НАСТРОЙКА МОДЕЛИ")
print("="*50)

# Загрузка модели
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Подготовка модели для k-bit обучения
model = prepare_model_for_kbit_training(model)

# УЛУЧШЕННАЯ конфигурация LoRA
lora_config = LoraConfig(
    r=16,  # Увеличил rank для лучшего обучения
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,  # Уменьшил dropout
    bias="none",
    task_type="CAUSAL_LM"
)

# Применение LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# УЛУЧШЕННЫЕ параметры обучения
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR / "model_output",
    num_train_epochs=5,  # Увеличил количество эпох
    per_device_train_batch_size=1,  # Уменьшил batch size для работы с длинными текстами
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_steps=100,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    learning_rate=1e-4,  # Уменьшил learning rate
    fp16=True,
    optim="paged_adamw_8bit",
    save_total_limit=3,
    load_best_model_at_end=True,
    report_to="none",
    gradient_checkpointing=True,  # Включил для экономии памяти
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print("\n" + "="*50)
print("НАЧАЛО ОБУЧЕНИЯ")
print("="*50)

# Обучение модели
trainer.train()

# Сохранение модели
model.save_pretrained(OUTPUT_DIR / "trained_model")
tokenizer.save_pretrained(OUTPUT_DIR / "trained_model")
print(f"\n✅ Модель сохранена в: {OUTPUT_DIR / 'trained_model'}")

# ==================================================
# УЛУЧШЕННАЯ ФУНКЦИЯ ДЛЯ ПРЕДСКАЗАНИЯ
# ==================================================
def predict_movie_ratings_improved(text_file_path, model, tokenizer, max_chars=20000):
    """
    УЛУЧШЕННОЕ предсказание рейтинга и возрастного рейтинга
    
    Args:
        text_file_path: Путь к текстовому файлу со сценарием
        model: Обученная модель
        tokenizer: Токенизатор
        max_chars: Максимальное количество символов для анализа
    """
    # Загрузка текста
    with open(text_file_path, 'r', encoding='utf-8') as f:
        script_text = f.read()
    
    # Очистка текста
    cleaned_text = TextPreprocessor.clean_script_text(script_text)
    
    # Обрезаем текст если слишком длинный
    if len(cleaned_text) > max_chars:
        cleaned_text = cleaned_text[:max_chars]
    
    # УЛУЧШЕННЫЙ промпт
    prompt = f"""ТЕКСТ СЦЕНАРИЯ ФИЛЬМА:

{cleaned_text}

ПРОАНАЛИЗИРУЙ ЭТОТ СЦЕНАРИЙ И ДАЙ ОЦЕНКУ:

РЕЙТИНГ (от 1.0 до 10.0, где 10.0 - отлично):
ВОЗРАСТНОЙ РЕЙТИНГ (выбери один: 0+, 6+, 12+, 16+, 18+):

ОТВЕТ:
РЕЙТИНГ ="""
    
    # Токенизация с вниманием к деталям
    inputs = tokenizer(
        prompt, 
        return_tensors="pt", 
        truncation=True, 
        max_length=2048,
        padding=True
    )
    
    # Перемещаем на GPU если доступно
    device = model.device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # УЛУЧШЕННАЯ генерация с лучшими параметрами
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.3,  # Низкая температура для более детерминированных ответов
            do_sample=False,   # Отключил sampling для более точных ответов
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    # Декодирование ответа
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Извлечение ответа (только часть после промпта)
    generated_text = answer[len(prompt):].strip()
    
    # УЛУЧШЕННЫЙ парсинг ответа
    rating = None
    age_rating = None
    
    # Поиск рейтинга в формате "РЕЙТИНГ = X.X"
    rating_patterns = [
        r'РЕЙТИНГ\s*=\s*([0-9]\.[0-9]|[0-9])',
        r'Рейтинг:\s*([0-9]\.[0-9]|[0-9])',
        r'([0-9]\.[0-9])/10',
        r'([0-9]\.[0-9])'
    ]
    
    for pattern in rating_patterns:
        match = re.search(pattern, generated_text, re.IGNORECASE)
        if match:
            try:
                rating = float(match.group(1))
                # Ограничиваем диапазон 1.0-10.0
                if rating < 1.0:
                    rating = 1.0
                elif rating > 10.0:
                    rating = 10.0
                break
            except:
                continue
    
    # Поиск возрастного рейтинга
    age_patterns = [
        r'ВОЗРАСТНОЙ РЕЙТИНГ\s*=\s*([0-9]+\+)',
        r'Возрастной рейтинг:\s*([0-9]+\+)',
        r'([0-9]+\+)\s*возраст',
        r'([0-9]+\+)'
    ]
    
    for pattern in age_patterns:
        match = re.search(pattern, generated_text, re.IGNORECASE)
        if match:
            age_rating = match.group(1)
            # Нормализация
            if age_rating in ['18+', '16+', '12+', '6+', '0+']:
                break
            else:
                # Если не стандартный, пробуем преобразовать
                if '18' in age_rating:
                    age_rating = '18+'
                elif '16' in age_rating:
                    age_rating = '16+'
                elif '12' in age_rating:
                    age_rating = '12+'
                elif '6' in age_rating:
                    age_rating = '6+'
                elif '0' in age_rating:
                    age_rating = '0+'
                break
    
    return {
        "filename": text_file_path.name,
        "predicted_rating": rating,
        "predicted_age_rating": age_rating,
        "full_response": generated_text,
        "prompt_used": prompt[:200] + "..." if len(prompt) > 200 else prompt
    }

# ==================================================
# ТЕСТИРОВАНИЕ НА ТЕСТОВЫХ ФАЙЛАХ (ТОЛЬКО NOINFO!)
# ==================================================
print("\n" + "="*50)
print("ТЕСТИРОВАНИЕ НА НОВЫХ ФАЙЛАХ (ТОЛЬКО NOINFO)")
print("="*50)

# Только папка noinfo
test_dirs = [TEST_TXT_DIR]

all_test_results = []

for test_dir in test_dirs:
    if test_dir.exists():
        print(f"\n📂 Тестирование файлов из: {test_dir}")
        
        # Поиск всех txt файлов
        test_files = list(test_dir.glob("*.txt"))
        
        print(f"Найдено файлов: {len(test_files)}")
        
        for i, test_file in enumerate(test_files, 1):
            try:
                print(f"\n┌─ Файл {i}/{len(test_files)}: {test_file.name}")
                result = predict_movie_ratings_improved(test_file, model, tokenizer, max_chars=15000)
                all_test_results.append(result)
                
                print(f"├─ Предсказанный рейтинг: ", end="")
                if result['predicted_rating']:
                    print(f"✅ {result['predicted_rating']:.1f}/10")
                else:
                    print(f"❌ Не определен")
                
                print(f"├─ Возрастной рейтинг: ", end="")
                if result['predicted_age_rating']:
                    print(f"✅ {result['predicted_age_rating']}")
                else:
                    print(f"❌ Не определен")
                
                print(f"└─ Ответ модели: {result['full_response'][:80]}...")
                
            except Exception as e:
                print(f"❌ Ошибка при обработке {test_file.name}: {e}")

# ==================================================
# АНАЛИЗ РЕЗУЛЬТАТОВ ТЕСТИРОВАНИЯ
# ==================================================
print("\n" + "="*50)
print("АНАЛИЗ РЕЗУЛЬТАТОВ ТЕСТИРОВАНИЯ")
print("="*50)

if all_test_results:
    # Создаем DataFrame для анализа
    results_df = pd.DataFrame(all_test_results)
    
    # Статистика
    successful_rating = results_df['predicted_rating'].notna().sum()
    successful_age = results_df['predicted_age_rating'].notna().sum()
    total_tests = len(results_df)
    
    print(f"\n📊 СТАТИСТИКА:")
    print(f"Всего протестировано файлов: {total_tests}")
    print(f"Успешно определено рейтингов: {successful_rating}/{total_tests} ({successful_rating/total_tests*100:.1f}%)")
    print(f"Успешно определено возрастных рейтингов: {successful_age}/{total_tests} ({successful_age/total_tests*100:.1f}%)")
    
    # Распределение предсказанных рейтингов
    if successful_rating > 0:
        ratings = results_df['predicted_rating'].dropna()
        print(f"\n📈 Распределение предсказанных рейтингов:")
        print(f"Средний: {ratings.mean():.1f}")
        print(f"Медиана: {ratings.median():.1f}")
        print(f"Минимальный: {ratings.min():.1f}")
        print(f"Максимальный: {ratings.max():.1f}")
    
    # Распределение возрастных рейтингов
    if successful_age > 0:
        age_counts = results_df['predicted_age_rating'].value_counts()
        print(f"\n👥 Распределение возрастных рейтингов:")
        for age, count in age_counts.items():
            print(f"  {age}: {count} файлов")
    
    # Сохранение результатов
    results_path = OUTPUT_DIR / "test_results.csv"
    results_df.to_csv(results_path, index=False, encoding=ENCODING)
    print(f"\n💾 Результаты сохранены в: {results_path}")
    
    # Вывод лучших и худших предсказаний
    print(f"\n🏆 ТОП-3 результата:")
    for i, (idx, row) in enumerate(results_df.iterrows()):
        if i >= 3:
            break
        print(f"\n{i+1}. {row['filename']}")
        print(f"   Рейтинг: {row['predicted_rating'] if row['predicted_rating'] else 'Нет'}")
        print(f"   Возрастной: {row['predicted_age_rating'] if row['predicted_age_rating'] else 'Нет'}")
else:
    print("❌ Нет результатов для анализа")

# ==================================================
# ИНТЕРФЕЙС ДЛЯ РУЧНОГО ТЕСТИРОВАНИЯ
# ==================================================
def interactive_testing_improved():
    """УЛУЧШЕННЫЙ интерактивный режим тестирования"""
    print("\n" + "="*50)
    print("🔧 ИНТЕРАКТИВНЫЙ РЕЖИМ ТЕСТИРОВАНИЯ")
    print("="*50)
    print("Введите путь к текстовому файлу со сценарием")
    print("Или введите 'exit' для выхода")
    print("Или 'demo' для тестового примера")
    
    while True:
        user_input = input("\n📁 Путь к файлу: ").strip()
        
        if user_input.lower() == 'exit':
            break
        elif user_input.lower() == 'demo':
            # Создаем демо-файл
            demo_text = """ИНТРО - НОЧЬ
Камера медленно приближается к окну квартиры. За окном - ночной город.
ВНУТРИ КВАРТИРЫ
АЛЕКС (30), программист, сидит за компьютером. На экране - код.
Он выглядит уставшим, но сосредоточенным.
АЛЕКС
(шепотом)
Еще немного... почти готово...
Он печатает последние строки кода, затем откидывается на спинку кресла.
АЛЕКС
(улыбаясь)
Сделано.
ФИНАЛЬНЫЕ ТИТРЫ"""
            
            demo_path = OUTPUT_DIR / "demo_script.txt"
            with open(demo_path, 'w', encoding='utf-8') as f:
                f.write(demo_text)
            
            print(f"\n📝 Создан демо-файл: {demo_path}")
            user_input = str(demo_path)
        
        file_path = Path(user_input)
        
        if not file_path.exists():
            print(f"❌ Файл не найден: {file_path}")
            continue
        
        try:
            print("\n" + "▬" * 50)
            print(f"🎬 АНАЛИЗ ФАЙЛА: {file_path.name}")
            print("▬" * 50)
            
            result = predict_movie_ratings_improved(file_path, model, tokenizer, max_chars=20000)
            
            print(f"\n📊 РЕЗУЛЬТАТЫ:")
            print(f"├─ Файл: {result['filename']}")
            
            if result['predicted_rating']:
                # Создаем визуальную шкалу рейтинга
                rating = result['predicted_rating']
                stars = "★" * int(rating)
                if rating - int(rating) >= 0.5:
                    stars += "½"
                stars = stars.ljust(10, '☆')
                
                print(f"├─ Рейтинг: {rating:.1f}/10")
                print(f"│  {stars}")
                
                # Интерпретация рейтинга
                if rating >= 9.0:
                    rating_desc = "ШЕДЕВР! 🏆"
                elif rating >= 8.0:
                    rating_desc = "ОТЛИЧНО! 👍"
                elif rating >= 7.0:
                    rating_desc = "ХОРОШО 👌"
                elif rating >= 6.0:
                    rating_desc = "НЕПЛОХО 🙂"
                elif rating >= 5.0:
                    rating_desc = "СРЕДНЕ 😐"
                else:
                    rating_desc = "НИЖЕ СРЕДНЕГО 👎"
                
                print(f"│  {rating_desc}")
            else:
                print(f"├─ Рейтинг: ❌ Не определен")
            
            if result['predicted_age_rating']:
                age = result['predicted_age_rating']
                print(f"├─ Возрастной рейтинг: {age}")
                
                # Описание возрастного рейтинга
                age_descriptions = {
                    '0+': "Для всех возрастов 👶",
                    '6+': "Для детей от 6 лет 🧒",
                    '12+': "Для подростков от 12 лет 🧑",
                    '16+': "Для молодежи от 16 лет 🧑‍🎓",
                    '18+': "Только для взрослых 🔞"
                }
                
                if age in age_descriptions:
                    print(f"│  {age_descriptions[age]}")
            else:
                print(f"├─ Возрастной рейтинг: ❌ Не определен")
            
            print(f"\n📝 ПОЛНЫЙ ОТВЕТ МОДЕЛИ:")
            print("─" * 40)
            print(result['full_response'])
            print("─" * 40)
            
            print(f"\n💡 СОВЕТ: ", end="")
            if result['predicted_rating'] and result['predicted_age_rating']:
                if result['predicted_rating'] >= 8.0 and result['predicted_age_rating'] == '18+':
                    print("Этот сценарий выглядит перспективным для взрослой аудитории!")
                elif result['predicted_rating'] >= 7.0:
                    print("Сценарий имеет хороший потенциал. Рекомендуется доработка.")
                else:
                    print("Сценарий требует существенной доработки.")
            else:
                print("Не удалось полностью проанализировать сценарий.")
            
            print("▬" * 50)
            
            # Сохранение результата
            save_result = input("\n💾 Сохранить результат? (да/нет): ").lower()
            if save_result in ['да', 'д', 'yes', 'y']:
                timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
                save_path = OUTPUT_DIR / f"analysis_{file_path.stem}_{timestamp}.txt"
                
                with open(save_path, 'w', encoding='utf-8') as f:
                    f.write(f"АНАЛИЗ СЦЕНАРИЯ: {file_path.name}\n")
                    f.write(f"Дата анализа: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
                    f.write("="*50 + "\n\n")
                    f.write(f"📊 РЕЙТИНГ: {result['predicted_rating'] if result['predicted_rating'] else 'Нет'}/10\n")
                    f.write(f"👥 ВОЗРАСТНОЙ РЕЙТИНГ: {result['predicted_age_rating'] if result['predicted_age_rating'] else 'Нет'}\n\n")
                    f.write("📝 ПОЛНЫЙ ОТВЕТ МОДЕЛИ:\n")
                    f.write("-"*40 + "\n")
                    f.write(result['full_response'] + "\n")
                    f.write("-"*40 + "\n")
                
                print(f"✅ Результат сохранен в: {save_path}")
            
        except Exception as e:
            print(f"❌ Ошибка: {e}")
            import traceback
            traceback.print_exc()

# ==================================================
# ФИНАЛЬНЫЙ ЭКСПОРТ И СОХРАНЕНИЕ
# ==================================================
print("\n" + "="*50)
print("✅ ОБУЧЕНИЕ ЗАВЕРШЕНО!")
print("="*50)

# Сохранение итоговой конфигурации
config = {
    "model_name": MODEL_NAME,
    "training_samples": len(train_df),
    "validation_samples": len(val_df),
    "min_text_length": MIN_TEXT_LENGTH,
    "max_text_length": MAX_TEXT_LENGTH,
    "avg_text_length": processed_df['text'].apply(len).mean(),
    "test_files_analyzed": len(all_test_results) if all_test_results else 0,
    "date_trained": pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    "training_notes": "Обучение на полных сценариях, улучшенные промпты, тестирование только на noinfo"
}

with open(OUTPUT_DIR / "training_config.json", "w", encoding=ENCODING) as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("📋 КОНФИГУРАЦИЯ:")
for key, value in config.items():
    print(f"  {key}: {value}")

print("\n🚀 МОДЕЛЬ ГОТОВА К ИСПОЛЬЗОВАНИЮ!")
print("\n📌 КОМАНДЫ ДЛЯ ИСПОЛЬЗОВАНИЯ:")
print("1. Для интерактивного тестирования: interactive_testing_improved()")
print("2. Для анализа конкретного файла:")
print("   result = predict_movie_ratings_improved('путь/к/файлу.txt', model, tokenizer)")
print("\n📂 РЕЗУЛЬТАТЫ СОХРАНЕНЫ В:")
print(f"   - Модель: {OUTPUT_DIR / 'trained_model'}")
print(f"   - Конфигурация: {OUTPUT_DIR / 'training_config.json'}")
print(f"   - Результаты тестов: {OUTPUT_DIR / 'test_results.csv' if all_test_results else 'Нет'}")

# Сохраняем пример использования в файл
usage_example = f"""
ИНСТРУКЦИЯ ПО ИСПОЛЬЗОВАНИЮ МОДЕЛИ
===================================

Модель обучена предсказывать рейтинг фильма (1.0-10.0) и возрастной рейтинг (0+, 6+, 12+, 16+, 18+)
по тексту сценария.

ИСПОЛЬЗОВАНИЕ:

1. Интерактивный режим:
   >>> interactive_testing_improved()
   
2. Программный анализ:
   >>> from transformers import AutoTokenizer, AutoModelForCausalLM
   >>> from peft import PeftModel
   >>> import torch
   
   # Загрузка модели
   >>> tokenizer = AutoTokenizer.from_pretrained("{OUTPUT_DIR / 'trained_model'}")
   >>> model = AutoModelForCausalLM.from_pretrained(
   ...     "{OUTPUT_DIR / 'trained_model'}",
   ...     device_map="auto",
   ...     torch_dtype=torch.float16
   ... )
   
   # Анализ файла
   >>> result = predict_movie_ratings_improved("путь/к/сценарию.txt", model, tokenizer)
   >>> print(f"Рейтинг: {{result['predicted_rating']:.1f}}")
   >>> print(f"Возрастной рейтинг: {{result['predicted_age_rating']}}")

ПАРАМЕТРЫ МОДЕЛИ:
{json.dumps(config, indent=2, ensure_ascii=False)}

Дата создания: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

with open(OUTPUT_DIR / "README.md", "w", encoding=ENCODING) as f:
    f.write(usage_example)

print(f"\n📖 Инструкция сохранена в: {OUTPUT_DIR / 'README.md'}")

# Запускаем интерактивный режим
print("\n" + "="*50)
print("🚀 ЗАПУСК ИНТЕРАКТИВНОГО РЕЖИМА...")
print("="*50)

interactive_testing_improved()

CUDA: True
NVIDIA GeForce RTX 5070 Ti
Конфигурация загружена!
ROOT: C:\Users\Дмитрий\Downloads\Ratings
Загружено 51 записей из scripts_ratingss.csv

Первые 3 записи:
                    filename                  title  year  kp_rating  \
0  13_причин_почему_Кино.txt  13_причин_почему_Кино  2017        7.3   
1    Игра_Престолов_Кино.txt    Игра_Престолов_Кино  2011        9.0   
2     8_миллиметров_Кино.txt     8_миллиметров_Кино  1999        7.1   

   imdb_rating notes age_rating_imdb age_rating_kp           english_title  
0          7.4    ok           TV-MA           16+    TH1RTEEN R3ASONS WHY  
1          9.2    ok           TV-MA           18+  A Song of Ice and Fire  
2          6.6    ok               R           18+            8 Millimeter  
Обрезан длинный текст: 8_миллиметров_Кино.txt (100000 символов)
Обрезан длинный текст: Джокер_Кино.txt (100000 символов)
Обрезан длинный текст: Довод_Кино.txt (100000 символов)
Обрезан длинный текст: Железный_человек_Кино.txt (100000 сим

c:\Users\Дмитрий\Downloads\Ratings\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Подготовка датасетов...


Map: 100%|██████████| 8/8 [00:00<00:00, 262.87 examples/s]


Размеры датасетов:
Обучающий: 43 примеров
Валидационный: 8 примеров

НАСТРОЙКА МОДЕЛИ
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820

НАЧАЛО ОБУЧЕНИЯ


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.



✅ Модель сохранена в: C:\Users\Дмитрий\Downloads\Ratings\Out\trained_model

ТЕСТИРОВАНИЕ НА НОВЫХ ФАЙЛАХ (ТОЛЬКО NOINFO)

📂 Тестирование файлов из: C:\Users\Дмитрий\Downloads\Ratings\datasets\noinfo
Найдено файлов: 3

┌─ Файл 1/3: Анатидаефобия_Кино.txt
├─ Предсказанный рейтинг: ❌ Не определен
├─ Возрастной рейтинг: ❌ Не определен
└─ Ответ модели: ...

┌─ Файл 2/3: Прокрастинация_Кино.txt
├─ Предсказанный рейтинг: ❌ Не определен
├─ Возрастной рейтинг: ❌ Не определен
└─ Ответ модели: ...

┌─ Файл 3/3: Ширванская_сказка_Кино.txt
├─ Предсказанный рейтинг: ❌ Не определен
├─ Возрастной рейтинг: ❌ Не определен
└─ Ответ модели: ...

АНАЛИЗ РЕЗУЛЬТАТОВ ТЕСТИРОВАНИЯ

📊 СТАТИСТИКА:
Всего протестировано файлов: 3
Успешно определено рейтингов: 0/3 (0.0%)
Успешно определено возрастных рейтингов: 0/3 (0.0%)

💾 Результаты сохранены в: C:\Users\Дмитрий\Downloads\Ratings\Out\test_results.csv

🏆 ТОП-3 результата:

1. Анатидаефобия_Кино.txt
   Рейтинг: Нет
   Возрастной: Нет

2. Прокрастинация_Кино.txt


NameError: name 'json' is not defined